# 17 — Readout Bayesian Optimization

Legacy experiment 52.

Automated Bayesian optimization of readout F×Q (fidelity × quality) over
readout amplitude, length, frequency, and SPA power.

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    ReadoutButterflyMeasurement,
    ReadoutGEDiscrimination,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="17_readout_bayesian_optimization",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

readout_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if readout_checkpoint is not None:
    print(f"Prior stage 16 status: {readout_checkpoint['status']}")

## 2. Bayesian Optimization Defaults

In [ ]:
APPLY_BEST = True

# Bayesian optimization
N_CALLS = 50
N_INITIAL_POINTS = 10
N_AVG_PER_EVAL = 5000

# Parameter bounds: (min, max)
BOUNDS_READOUT_AMP = (0.05, 0.5)
BOUNDS_READOUT_LEN_NS = (200, 2000)
BOUNDS_READOUT_FREQ_OFFSET_MHZ = (-2.0, 2.0)
BOUNDS_SPA_POWER_DBM = (-30, 0)

print("Bayesian optimization settings:")
print(f"  n_calls: {N_CALLS}, n_initial_points: {N_INITIAL_POINTS}")
print(f"  n_avg per eval: {N_AVG_PER_EVAL}")
print(f"  Bounds:")
print(f"    readout_amp: {BOUNDS_READOUT_AMP}")
print(f"    readout_len_ns: {BOUNDS_READOUT_LEN_NS}")
print(f"    readout_freq_offset_MHz: {BOUNDS_READOUT_FREQ_OFFSET_MHZ}")
print(f"    spa_power_dBm: {BOUNDS_SPA_POWER_DBM}")

## 3. Define Objective Function

Return negative F×Q (since `gp_minimize` minimizes).

In [ ]:
def readout_fxq_objective(params):
    """Evaluate readout F×Q for a given parameter set.

    Returns negative F×Q (scikit-optimize minimizes).
    """
    amp, length_ns, freq_offset_mhz, spa_power_dbm = params

    # Apply trial parameters to session
    session.set_readout_trial_params(
        amplitude=amp,
        length_ns=int(length_ns),
        frequency_offset_mhz=freq_offset_mhz,
        spa_power_dbm=spa_power_dbm,
    )

    # Run butterfly measurement to get F×Q
    butterfly = ReadoutButterflyMeasurement(session)
    result = butterfly.run(n_avg=N_AVG_PER_EVAL)
    analysis = butterfly.analyze(result, update_calibration=False)

    fxq = analysis.metrics.get("FxQ", 0.0)
    print(f"    amp={amp:.3f}, len={int(length_ns)}ns, Δf={freq_offset_mhz:+.2f}MHz, SPA={spa_power_dbm:.1f}dBm → F×Q={fxq:.4f}")

    return -fxq  # minimize negative F×Q

print("Objective function defined.")

## 4. Run Bayesian Optimization — Exp 52

In [ ]:
from skopt import gp_minimize

bounds = [
    BOUNDS_READOUT_AMP,
    BOUNDS_READOUT_LEN_NS,
    BOUNDS_READOUT_FREQ_OFFSET_MHZ,
    BOUNDS_SPA_POWER_DBM,
]

opt_result = gp_minimize(
    readout_fxq_objective,
    dimensions=bounds,
    n_calls=N_CALLS,
    n_initial_points=N_INITIAL_POINTS,
    random_state=42,
)

best_amp, best_len_ns, best_freq_offset, best_spa_power = opt_result.x
best_fxq = -opt_result.fun

print(f"\nBest F×Q: {best_fxq:.4f}")
print(f"  Readout amplitude: {best_amp:.3f}")
print(f"  Readout length: {int(best_len_ns)} ns")
print(f"  Frequency offset: {best_freq_offset:+.2f} MHz")
print(f"  SPA power: {best_spa_power:.1f} dBm")

## 5. Convergence and Parameter Landscape

In [ ]:
from skopt.plots import plot_convergence, plot_objective

fig, ax = plt.subplots(figsize=(8, 5))
plot_convergence(opt_result, ax=ax)
ax.set_title("Bayesian Optimization Convergence")
plt.tight_layout()
plt.show()

fig = plot_objective(
    opt_result,
    dimensions=["Amplitude", "Length (ns)", "Δf (MHz)", "SPA (dBm)"],
)
plt.suptitle("Parameter Landscape")
plt.tight_layout()
plt.show()

## 6. Apply Best Parameters

In [ ]:
if APPLY_BEST:
    proposed_patch_ops = [
        {"op": "replace", "path": "/readout/amplitude", "value": float(best_amp)},
        {"op": "replace", "path": "/readout/length_ns", "value": int(best_len_ns)},
        {"op": "replace", "path": "/readout/frequency_offset_mhz", "value": float(best_freq_offset)},
        {"op": "replace", "path": "/spa/power_dbm", "value": float(best_spa_power)},
    ]

    bayes_patch, _, bayes_apply = preview_or_apply_patch_ops(
        session,
        reason="Bayesian readout optimization",
        proposed_patch_ops=proposed_patch_ops,
        apply=APPLY_BEST,
    )

    if bayes_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)

    print("Best parameters applied.")
else:
    print("APPLY_BEST=False — best parameters not applied.")

## 7. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="17_readout_bayesian_optimization",
    status="calibrated" if APPLY_BEST else "characterized",
    summary=f"Bayesian optimization of readout F×Q. Best F×Q={best_fxq:.4f}.",
    consumed_inputs={
        "n_calls": N_CALLS,
        "n_initial_points": N_INITIAL_POINTS,
        "n_avg_per_eval": N_AVG_PER_EVAL,
        "bounds_readout_amp": list(BOUNDS_READOUT_AMP),
        "bounds_readout_len_ns": list(BOUNDS_READOUT_LEN_NS),
        "bounds_readout_freq_offset_mhz": list(BOUNDS_READOUT_FREQ_OFFSET_MHZ),
        "bounds_spa_power_dbm": list(BOUNDS_SPA_POWER_DBM),
    },
    persisted_outputs={
        "best_fxq": best_fxq,
        "best_readout_amp": float(best_amp),
        "best_readout_len_ns": int(best_len_ns),
        "best_freq_offset_mhz": float(best_freq_offset),
        "best_spa_power_dbm": float(best_spa_power),
        "applied": APPLY_BEST,
    },
    advisory_outputs={},
    next_stage="18_active_reset_benchmarking",
    notes=[
        "Depends on scikit-optimize (skopt) and ipywidgets.",
        "Objective uses ReadoutButterflyMeasurement for F×Q metric.",
    ],
    metrics={
        "best_fxq": best_fxq,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")